In [ ]:
# Remove unneeded objects from memory
rm(list=ls())

In [1]:
# Load packages
library(Matrix)
library(Seurat)
library(ggplot2)
library(ggsci)
library(dplyr)
library(pheatmap)
library(RColorBrewer)

Attaching SeuratObject


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
# Load data
load('/home/jovyan/Data/GSE124872_raw_counts_single_cell.RData')
meta<-read.csv('/home/jovyan/Data/GSE124872_Angelidis_2018_metadata.csv')

In [ ]:
# Inspect and transform loaded data
raw_counts[1:5, 1:5]
head(meta)

meta$bc<-sapply(strsplit(split=':', meta$X), '[', 3)

In [ ]:
# Create analysis object
seurat<-CreateSeuratObject(counts=as.matrix(raw_counts), min.cells=3, min.features=50)

In [ ]:
# Add metadata
seurat$barcodes<-sapply(strsplit(split=':', rownames(seurat@meta.data)), '[', 2)
seurat$mouseid<-sapply(strsplit(split=':', rownames(seurat@meta.data)), '[', 1)
seurat$age<-sapply(strsplit(split='_', seurat$mouseid), '[', 1)
seurat$annotation<-meta$celltype[match(seurat@meta.data$barcodes, meta$bc)]

head(seurat@meta.data)

In [ ]:
# Decide on inflection point
df<-data.frame(row.names=rownames(seurat@meta.data),
               cell=rownames(seurat@meta.data),
               Gene=seurat$nFeature_RNA,
               UMI=seurat$nCount_RNA)

df<-df[order(df$Gene, decreasing=T),]
df$index<-1:nrow(df)

options(repr.plot.width=10, repr.plot.height=6)
ggplot(df, aes(index, Gene))+
  geom_point(size=0.1, show.legend=F)+
  scale_y_log10()+
  scale_x_log10()+
  geom_hline(aes(yintercept=200), color='blue')+
  geom_hline(aes(yintercept=100), color='red')+
  ylab('Gene numbers')+
  xlab('cells sorted by gene count')+
  theme_bw()

ggplot(df, aes(Gene, UMI))+
  geom_point(size=0.1, show.legend=F)+
  scale_y_log10()+
  scale_x_log10()+
  geom_vline(aes(xintercept=100), color='red')+
  geom_vline(aes(xintercept=200), color='blue')+
  ylab('UMI numbers')+
  xlab('Gene numbers')+
  theme_bw()

In [ ]:
# Filter on number of genes
quantile(seurat$nFeature_RNA, probs=c(0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1))
VlnPlot(object=seurat, features=c("nFeature_RNA", "nCount_RNA"), log=F, ncol=2)

seurat<-subset(seurat, subset=nFeature_RNA>=200)

In [ ]:
# Filter on mitochondrial gene content
seurat[["percent.mt"]]<-PercentageFeatureSet(object=seurat, pattern="^mt-")
quantile(seurat$nFeature_RNA, probs=c(0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1))
VlnPlot(object=seurat, features=c("percent.mt"), log=F, ncol=1)

seurat<-subset(seurat, subset=percent.mt<=10)

In [ ]:
# QC analysis
Idents(seurat)<-'orig.ident'
VlnPlot(object=seurat, features=c("nFeature_RNA", "nCount_RNA", "percent.mt"), log=F, ncol=3)

In [ ]:
# Normalisation 
seurat<-NormalizeData(object=seurat, normalization.method="LogNormalize", scale.factor=10000)

In [ ]:
# Find variable genes
seurat<-FindVariableFeatures(object=seurat, selection.method="vst", nfeatures=2000)

In [ ]:
# Scale data
seurat<-ScaleData(object=seurat)

In [ ]:
# Perform linear reduction    
seurat<-RunPCA(object=seurat, features=seurat@assays$RNA@var.features, verbose=F, npcs=50)
print(x=seurat[["pca"]], dims=1:15, nfeatures=5)

In [ ]:
# Visualize PCA attributes
options(repr.plot.width=12, repr.plot.height=20)
VizDimLoadings(object=seurat, dims=1:9, reduction="pca")

options(repr.plot.width=9, repr.plot.height=6)
DimPlot(object=seurat, reduction="pca")

In [ ]:
# Decide on which PCs to keep
ElbowPlot(object=seurat, ndims=50)

In [ ]:
# Find k-nearest neighbours
seurat<-FindNeighbors(object=seurat, reduction='pca', dims=1:20)

In [ ]:
# Find clusters
seurat<-FindClusters(object=seurat, resolution=0.4, group.singletons=T)

In [ ]:
# Run UMAP
#seurat<-RunUMAP(object=seurat, reduction="pca", dims=1:20, verbose=F)

options(repr.plot.width=12, repr.plot.height=8)
DimPlot(object=seurat, reduction='umap', label=T)

In [ ]:
# Look for potential batches
options(repr.plot.width=12, repr.plot.height=8)
DimPlot(seurat, group.by='mouseid') + scale_color_jco()
DimPlot(seurat, group.by='age') + scale_color_npg()

In [ ]:
# Calculate DE genes
deg<-FindAllMarkers(seurat, min.pct=0.20, logfc.threshold=0.25, test.use='MAST', only.pos=T)
deg<-deg[deg$p_val_adj<=0.05,]
deg<-deg %>% arrange(cluster, -avg_log2FC)
deg5<-deg %>% group_by(cluster) %>% top_n(n=5, wt=avg_log2FC)

In [ ]:
# Visualise top 5 cluster markers
options(repr.plot.width=20, repr.plot.height=8)
DotPlot(object=seurat, features=unique(deg5$gene), cols='RdBu', dot.scale=6, dot.min=0.1) + RotatedAxis()

In [ ]:
# Annotate the object
Idents(seurat)<-'RNA_snn_res.0.4'

seurat<-RenameIdents(object=seurat,
                     `0` = "",
                     `1` = "",
                     `2` = "", 
                     `3` = "", 
                     `4` = "", 
                     `5` = "",
                     `6` = "",
                     `7` = "",              
                     `8` = "", 
                     `9` = "", 
                     `10` = "",
                     `11` = "",
                     `12` = "",
                     `13` = "",
                     `14` = "",
                     `15` = "",
                     `16` = "",
                     `17` = "",
                     `18` = "")

seurat$new_annotation<-Idents(seurat)

In [ ]:
# Inspect published annotation
options(repr.plot.width=15, repr.plot.height=8)
DimPlot(seurat, group.by='annotation', label=T) + theme(legend.position="none")

In [ ]:
# Compare the two annotations
df<-table(seurat$annotation, seurat$new_annotation)

options(repr.plot.width=8, repr.plot.height=8)
pheatmap(df, scale='row', clustering_distance_rows='correlation', cluster_rows=T, cluster_cols=F, cellwidth=15, cellheight=15, color = colorRampPalette(rev(brewer.pal(n=5, name="RdBu")))(20))